In [ ]:
%cd /kaggle/working

REPO_BRANCH = "main"

!rm -rf temporal-recsys
!git clone --branch {REPO_BRANCH} --single-branch https://github.com/dshhrv/temporal-recsys.git

%cd /kaggle/working/temporal-recsys

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

from src.tgn_model import TGN

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_EPOCHS = 10
PATIENCE = 2
VALID_METRIC = "Recall@10"
TRAIN_BATCH_SIZE = 1000
EVAL_BATCH_SIZE = 64
NUM_NEGATIVE_ITEMS = 5
LEARNING_RATE = 0.0005
WEIGHT_DECAY = 0.0

EMBEDDING_DIM = 64
MEMORY_DIM = 64
TIME_DIM = 64
EDGE_DIM = 1
NUM_NEIGHBORS = 10
NUM_ATTENTION_HEADS = 1
ITEM_CHUNK_SIZE = 256

TIME_FIELD = "time_days"
EDGE_FEATURE_FIELD = "edge_feat_0"

DATA_DIR = "/kaggle/input/datasets/dshhrv/tgn-ml1m1"
INTERACTIONS_PATH = f"{DATA_DIR}/interactions.csv"
META_PATH = f"{DATA_DIR}/meta.json"

OUTPUT_DIR = Path("/kaggle/working/tgn_bpr_results")
EPOCH_METRICS_PATH = OUTPUT_DIR / "tgn_bpr_epoch_metrics.csv"
TEST_METRICS_PATH = OUTPUT_DIR / "tgn_bpr_test_metrics.csv"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "tgn_bpr_best.pt"
LAST_CHECKPOINT_PATH = OUTPUT_DIR / "tgn_bpr_last.pt"
RUN_CONFIG_PATH = OUTPUT_DIR / "tgn_bpr_run_config.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"Epochs: {MAX_EPOCHS}")
print(f"Train batch size: {TRAIN_BATCH_SIZE}")

In [ ]:
interactions = pd.read_csv(INTERACTIONS_PATH).sort_values("event_id", kind="stable").reset_index(drop=True)

required_columns = {
    "event_id",
    "src",
    "dst",
    "split",
    TIME_FIELD,
    EDGE_FEATURE_FIELD,
}

missing_columns = required_columns - set(interactions.columns)
if missing_columns:
    raise RuntimeError(
        f"В interactions.csv нет {sorted(missing_columns)}. "
        "Перезапусти новый preprocess_tgn_ml1m.py и загрузи обновленный data/tgn_ml1m в Kaggle."
    )

with open(META_PATH, "r", encoding="utf-8") as file:
    metadata = json.load(file)

NUM_USERS = int(metadata["n_users"])
NUM_ITEMS = int(metadata["n_items"])

interactions["user_idx"] = interactions["src"].astype(np.int64)
interactions["item_idx"] = (interactions["dst"] - NUM_USERS).astype(np.int64)

if not interactions["user_idx"].between(0, NUM_USERS - 1).all():
    raise RuntimeError("Некорректные user node ids в interactions.csv.")

if not interactions["item_idx"].between(0, NUM_ITEMS - 1).all():
    raise RuntimeError(
        "Некорректные item ids. В модель должны идти локальные item ids dst - n_users, "
        "а не глобальные dst."
    )

TRAIN = interactions[interactions["split"] == "train"].copy()
VALID = interactions[interactions["split"] == "valid"].copy()
TEST = interactions[interactions["split"] == "test"].copy()

warm_item_values = np.sort(TRAIN["item_idx"].unique())
warm_item_set = set(warm_item_values.tolist())

if len(warm_item_values) != NUM_ITEMS:
    raise RuntimeError(
        "В каталоге остались cold items. Нужен новый preprocessing: global 80/10/10, "
        "потом удалить cold items из valid/test и remap каталога."
    )

if not VALID["item_idx"].isin(warm_item_set).all() or not TEST["item_idx"].isin(warm_item_set).all():
    raise RuntimeError("В valid/test остались cold items. Перегенерируй data/tgn_ml1m.")

catalog_item_ids = torch.arange(NUM_ITEMS, dtype=torch.long, device=DEVICE)
cold_item_ids = torch.empty(0, dtype=torch.long, device=DEVICE)

print(f"Train: {len(TRAIN):,}")
print(f"Valid: {len(VALID):,}")
print(f"Test: {len(TEST):,}")
print(f"Catalog: {NUM_ITEMS:,} items")
print(f"Edge feature: {metadata.get('edge_feature', EDGE_FEATURE_FIELD)}")

In [ ]:
def make_temporal_batches(frame, batch_size):
    user_ids = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long)
    item_ids = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long)
    timestamps = torch.tensor(frame[TIME_FIELD].to_numpy(dtype=np.float32), dtype=torch.float32)
    edge_features = torch.tensor(frame[EDGE_FEATURE_FIELD].to_numpy(dtype=np.float32), dtype=torch.float32).unsqueeze(-1)
    times = timestamps.numpy()

    batches = []
    start = 0

    while start < len(frame):
        end = min(start + batch_size, len(frame))

        while end < len(frame) and times[end] == times[end - 1]:
            end += 1

        batches.append((user_ids[start:end], item_ids[start:end], timestamps[start:end], edge_features[start:end]))
        start = end

    return batches

train_loader = make_temporal_batches(TRAIN, TRAIN_BATCH_SIZE)
valid_loader = make_temporal_batches(VALID, EVAL_BATCH_SIZE)
test_loader = make_temporal_batches(TEST, EVAL_BATCH_SIZE)

train_seen = [set() for _ in range(NUM_USERS)]

for user_id, item_id in zip(TRAIN["user_idx"].to_numpy(), TRAIN["item_idx"].to_numpy()):
    train_seen[int(user_id)].add(int(item_id))

def copy_seen(seen_items):
    return [items.copy() for items in seen_items]

def add_frame_to_seen(seen_items, frame):
    frame_users = frame["user_idx"].to_numpy()
    frame_items = frame["item_idx"].to_numpy()

    for user_id, item_id in zip(frame_users, frame_items):
        seen_items[int(user_id)].add(int(item_id))

print(f"Train batches: {len(train_loader):,}")
print(f"Valid batches: {len(valid_loader):,}")
print(f"Test batches: {len(test_loader):,}")


In [ ]:
model = TGN(
    NUM_USERS,
    NUM_ITEMS,
    node_dim=EMBEDDING_DIM,
    memory_dim=MEMORY_DIM,
    time_dim=TIME_DIM,
    edge_dim=EDGE_DIM,
    num_neighbors=NUM_NEIGHBORS,
    num_heads=NUM_ATTENTION_HEADS,
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history = []
best_valid_score = -float("inf")
best_epoch = None
bad_epochs = 0

run_config = {
    "seed": SEED,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "valid_metric": VALID_METRIC,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "num_negative_items": NUM_NEGATIVE_ITEMS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "embedding_dim": EMBEDDING_DIM,
    "memory_dim": MEMORY_DIM,
    "time_dim": TIME_DIM,
    "edge_dim": EDGE_DIM,
    "num_neighbors": NUM_NEIGHBORS,
    "num_attention_heads": NUM_ATTENTION_HEADS,
    "item_chunk_size": ITEM_CHUNK_SIZE,
    "time_field": TIME_FIELD,
    "edge_feature_field": EDGE_FEATURE_FIELD,
    "data_dir": DATA_DIR,
}

with open(RUN_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

def copy_model_state_dict():
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

def make_checkpoint(epoch, row, valid_metrics, is_best, model_state_dict):
    return {
        "epoch": epoch,
        "model_state_dict": model_state_dict,
        "optimizer_state_dict": optimizer.state_dict(),
        "valid_metrics": valid_metrics,
        "best_valid_score": best_valid_score,
        "best_epoch": best_epoch,
        "history": history,
        "config": run_config,
        "is_best": is_best,
        "last_epoch_metrics": row,
        "checkpoint_state": "after_train_before_validation",
    }

def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

print(f"Early stopping metric: {VALID_METRIC}")
print(f"Patience: {PATIENCE}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")


In [ ]:
TOPK = [5, 10, 20]
MAX_K = max(TOPK)

def update_metrics(totals, scores, targets):
    top_items = scores.topk(MAX_K, dim=1).indices
    matches = top_items.eq(targets.unsqueeze(1)).float()
    ranks = torch.arange(1, MAX_K + 1, device=scores.device, dtype=scores.dtype).unsqueeze(0)

    for k in TOPK:
        matches_at_k = matches[:, :k]
        ranks_at_k = ranks[:, :k]

        totals[f"Recall@{k}"] += matches_at_k.any(dim=1).float().sum().item()
        totals[f"NDCG@{k}"] += (matches_at_k / torch.log2(ranks_at_k + 1.0)).sum(dim=1).sum().item()
        totals[f"MRR@{k}"] += (matches_at_k / ranks_at_k).sum(dim=1).sum().item()

    totals["count"] += targets.size(0)

def seen_before_current_events(user_ids, item_ids, timestamps, seen_items):
    users = user_ids.detach().cpu().tolist()
    items = item_ids.detach().cpu().tolist()
    times = timestamps.detach().cpu().tolist()

    row_seen = [None] * len(users)
    start = 0

    while start < len(users):
        end = start + 1

        while end < len(users) and times[end] == times[start]:
            end += 1

        for row in range(start, end):
            row_seen[row] = seen_items[users[row]].copy()

        for row in range(start, end):
            seen_items[users[row]].add(items[row])

        start = end

    return row_seen

def mask_unavailable_items(scores, targets, row_seen):
    if cold_item_ids.numel() > 0:
        scores[:, cold_item_ids] = -torch.inf

    target_items = targets.detach().cpu().tolist()

    for row, (target_item_id, seen) in enumerate(zip(target_items, row_seen)):
        blocked = seen - {target_item_id}

        if blocked:
            blocked_ids = torch.tensor(list(blocked), dtype=torch.long, device=DEVICE)
            scores[row, blocked_ids] = -torch.inf

@torch.no_grad()
def evaluate(loader, seen_items):
    model.eval()

    totals = {"count": 0}

    for k in TOPK:
        totals[f"Recall@{k}"] = 0.0
        totals[f"NDCG@{k}"] = 0.0
        totals[f"MRR@{k}"] = 0.0

    for user_ids, item_ids, timestamps, edge_features in tqdm(loader, leave=False):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)

        model.flush_pending_messages()

        scores = model.score_catalog(
            user_ids,
            timestamps,
            catalog_item_ids,
            item_chunk_size=ITEM_CHUNK_SIZE,
        )

        row_seen = seen_before_current_events(user_ids, item_ids, timestamps, seen_items)
        mask_unavailable_items(scores, item_ids, row_seen)
        update_metrics(totals, scores, item_ids)

        model.store_batch(user_ids, item_ids, timestamps, edge_features)

    model.flush_pending_messages()
    model.detach_state()

    return {
        name: value / totals["count"]
        for name, value in totals.items()
        if name != "count"
    }

@torch.no_grad()
def warm_up(loader):
    model.eval()

    for user_ids, item_ids, timestamps, edge_features in tqdm(loader, leave=False):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)

        model.flush_pending_messages()
        model.store_batch(user_ids, item_ids, timestamps, edge_features)

    model.flush_pending_messages()
    model.detach_state()

In [ ]:
def draw_batch_negative(batch_item_pool, blocked):
    for _ in range(32):
        candidate = int(batch_item_pool[np.random.randint(len(batch_item_pool))])

        if candidate not in blocked:
            return candidate

    candidates = [int(item_id) for item_id in batch_item_pool if int(item_id) not in blocked]

    if not candidates:
        return None

    return candidates[np.random.randint(len(candidates))]

def draw_batch_negatives(batch_item_pool, blocked):
    negative_items = []
    blocked_with_drawn = set(blocked)

    for _ in range(NUM_NEGATIVE_ITEMS):
        negative_item_id = draw_batch_negative(batch_item_pool, blocked_with_drawn)

        if negative_item_id is None:
            if not negative_items:
                return None

            negative_item_id = negative_items[np.random.randint(len(negative_items))]
        else:
            blocked_with_drawn.add(negative_item_id)

        negative_items.append(negative_item_id)

    return negative_items

def sample_batch_negatives(user_ids, item_ids, timestamps, seen_train_items, check_asserts=False):
    users = user_ids.detach().cpu().numpy()
    positives = item_ids.detach().cpu().numpy()
    times = timestamps.detach().cpu().numpy()
    batch_item_pool = np.unique(positives).astype(np.int64)
    batch_item_pool_set = set(batch_item_pool.tolist())
    valid_rows = []
    negative_items = []
    skipped_examples = 0
    start = 0

    while start < len(users):
        end = start + 1

        while end < len(users) and times[end] == times[start]:
            end += 1

        positives_at_time = {}

        for row in range(start, end):
            positives_at_time.setdefault(int(users[row]), set()).add(int(positives[row]))

        for row in range(start, end):
            user_id = int(users[row])
            blocked = seen_train_items[user_id] | positives_at_time[user_id]
            negative_item_ids = draw_batch_negatives(batch_item_pool, blocked)

            if negative_item_ids is None:
                skipped_examples += 1
                continue

            if check_asserts:
                assert len(negative_item_ids) == NUM_NEGATIVE_ITEMS
                assert all(negative_item_id in batch_item_pool_set for negative_item_id in negative_item_ids)
                assert all(negative_item_id not in blocked for negative_item_id in negative_item_ids)

            valid_rows.append(row)
            negative_items.append(negative_item_ids)

        for user_id, positive_items in positives_at_time.items():
            seen_train_items[user_id].update(positive_items)

        start = end

    valid_rows = torch.tensor(valid_rows, dtype=torch.long, device=DEVICE)

    if negative_items:
        negative_items = torch.tensor(negative_items, dtype=torch.long, device=DEVICE)
    else:
        negative_items = torch.empty((0, NUM_NEGATIVE_ITEMS), dtype=torch.long, device=DEVICE)

    return valid_rows, negative_items, skipped_examples

def train_one_epoch(epoch):
    model.train()

    seen_train_items = [set() for _ in range(NUM_USERS)]
    total_loss = 0.0
    total_examples = 0
    skipped_examples = 0

    for batch_index, (user_ids, item_ids, timestamps, edge_features) in enumerate(tqdm(train_loader, leave=False)):
        user_ids = user_ids.to(DEVICE, non_blocking=True)
        item_ids = item_ids.to(DEVICE, non_blocking=True)
        timestamps = timestamps.to(DEVICE, non_blocking=True)
        edge_features = edge_features.to(DEVICE, non_blocking=True)

        valid_rows, negative_item_ids, skipped_in_batch = sample_batch_negatives(user_ids, item_ids, timestamps, seen_train_items, check_asserts=(epoch == 1 and batch_index == 0))
        skipped_examples += skipped_in_batch

        model.flush_pending_messages()

        if valid_rows.numel() > 0:
            filtered_user_ids = user_ids.index_select(0, valid_rows)
            filtered_item_ids = item_ids.index_select(0, valid_rows)
            filtered_timestamps = timestamps.index_select(0, valid_rows)

            optimizer.zero_grad(set_to_none=True)
            positive_scores, negative_scores = model(filtered_user_ids, filtered_item_ids, negative_item_ids, filtered_timestamps)
            loss = model.bpr_loss(positive_scores, negative_scores)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * filtered_item_ids.size(0)
            total_examples += filtered_item_ids.size(0)

        model.store_batch(user_ids, item_ids, timestamps, edge_features)
        model.detach_state()

    model.flush_pending_messages()
    model.detach_state()

    average_loss = total_loss / total_examples if total_examples else float("nan")

    return average_loss, total_examples, skipped_examples


In [ ]:
run_started = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_started = time.perf_counter()

    model.reset_state()

    train_started = time.perf_counter()
    train_loss, valid_bpr_pairs, skipped_bpr_examples = train_one_epoch(epoch)
    train_seconds = time.perf_counter() - train_started
    train_state_dict = copy_model_state_dict()

    validation_started = time.perf_counter()
    valid_metrics = evaluate(valid_loader, copy_seen(train_seen))
    validation_seconds = time.perf_counter() - validation_started

    epoch_seconds = time.perf_counter() - epoch_started
    valid_score = valid_metrics[VALID_METRIC]
    is_best = valid_score > best_valid_score

    if is_best:
        best_valid_score = valid_score
        best_epoch = epoch
        bad_epochs = 0
    else:
        bad_epochs += 1

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        **valid_metrics,
        "valid_bpr_pairs": valid_bpr_pairs,
        "skipped_bpr_examples": skipped_bpr_examples,
        "is_best": is_best,
        "best_epoch": best_epoch,
        "best_valid_score": best_valid_score,
        "bad_epochs": bad_epochs,
        "train_seconds": train_seconds,
        "validation_seconds": validation_seconds,
        "epoch_seconds": epoch_seconds,
        "cumulative_seconds": time.perf_counter() - run_started,
    }

    history.append(row)
    pd.DataFrame(history).to_csv(EPOCH_METRICS_PATH, index=False)

    checkpoint = make_checkpoint(epoch, row, valid_metrics, is_best, train_state_dict)
    torch.save(checkpoint, LAST_CHECKPOINT_PATH)

    if is_best:
        torch.save(checkpoint, BEST_CHECKPOINT_PATH)

    if epoch == 1:
        print(f"BPR negative sampling check | valid positives {valid_bpr_pairs:,} | BPR pairs {valid_bpr_pairs * NUM_NEGATIVE_ITEMS:,} | negatives per positive {NUM_NEGATIVE_ITEMS} | skipped examples {skipped_bpr_examples:,}")

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss {train_loss:.5f} | "
        f"Recall@5 {valid_metrics['Recall@5']:.5f} | "
        f"Recall@10 {valid_metrics['Recall@10']:.5f} | "
        f"Recall@20 {valid_metrics['Recall@20']:.5f} | "
        f"best_epoch {best_epoch} | bad_epochs {bad_epochs}"
    )

    if bad_epochs >= PATIENCE:
        print(f"Early stopping: {VALID_METRIC} did not improve for {PATIENCE} epoch(s).")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(EPOCH_METRICS_PATH, index=False)

print(f"Training finished in {(time.perf_counter() - run_started) / 3600:.2f} h")
print(f"Best epoch: {best_epoch} | best {VALID_METRIC}: {best_valid_score:.6f}")
print(f"Best checkpoint saved to: {BEST_CHECKPOINT_PATH}")
print(f"Last checkpoint saved to: {LAST_CHECKPOINT_PATH}")

best_checkpoint = load_checkpoint(BEST_CHECKPOINT_PATH)
model.load_state_dict(best_checkpoint["model_state_dict"])

valid_seen = copy_seen(train_seen)
valid_recheck = evaluate(valid_loader, valid_seen)
print("\nValidation recheck from best checkpoint")
print("Stored best valid:", best_checkpoint["valid_metrics"])
print("Recomputed valid:", valid_recheck)

model.load_state_dict(best_checkpoint["model_state_dict"])
warm_up(valid_loader)

test_seen = copy_seen(train_seen)
add_frame_to_seen(test_seen, VALID)

test_started = time.perf_counter()
test_metrics = evaluate(test_loader, test_seen)
test_seconds = time.perf_counter() - test_started

test_results = pd.DataFrame(
    [
        {
            "best_epoch": int(best_checkpoint["epoch"]),
            f"best_valid_{VALID_METRIC}": float(best_checkpoint["valid_metrics"][VALID_METRIC]),
            **test_metrics,
            "test_seconds": test_seconds,
        }
    ]
)
test_results.to_csv(TEST_METRICS_PATH, index=False)

print("\nTest metrics from best validation checkpoint")
print(test_results)
print(f"Test time: {test_seconds / 3600:.2f} h")


In [ ]:
history_df

In [ ]:
test_results

In [ ]:
for path in [EPOCH_METRICS_PATH, TEST_METRICS_PATH, BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH, RUN_CONFIG_PATH]:
    if path.exists():
        print(path)
